In [40]:
import pandas as pd
import json
from datetime import datetime
import re 
import seaborn as sns
from wordcloud import WordCloud 
from nltk.stem.wordnet import WordNetLemmatizer
import collections
from nltk import bigrams
import nltk
from nltk.corpus import stopwords

Sprawdzenie nazw kolumn i struktury danych, w celu wyfiltrowania potrzebnych kolumn w późniejszym etapie

In [ ]:
# sample_hp = pd.read_json(
#     "data/hiphopheads_comments",
#     lines=True,
#     nrows=1
# )

# print(sample_hp.info())

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   author_flair_css_class  0 non-null      float64
 1   link_id                 1 non-null      str    
 2   retrieved_on            1 non-null      int64  
 3   controversiality        1 non-null      int64  
 4   score                   1 non-null      int64  
 5   author_flair_text       0 non-null      float64
 6   name                    1 non-null      str    
 7   distinguished           0 non-null      float64
 8   body                    1 non-null      str    
 9   score_hidden            1 non-null      bool   
 10  ups                     1 non-null      int64  
 11  subreddit_id            1 non-null      str    
 12  parent_id               1 non-null      str    
 13  id                      1 non-null      str    
 14  edited                  1 non-null      bool   
 15  auth

dla komentarzy 21 kolumn, z nich będę potrzebować 11:
- link_id
- controversiality
- score
- distinguished
- body
- created_utc
- subreddit
- parent_id
- id
-author
- gilded

In [ ]:
# sample_hs = pd.read_json(
#     "data/hiphopheads_submissions",
#     lines=True,
#     nrows=1
# )

# print(sample_hs.info())

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 57 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   archived                       1 non-null      bool   
 1   author                         1 non-null      str    
 2   author_flair_background_color  0 non-null      float64
 3   author_flair_css_class         1 non-null      str    
 4   author_flair_richtext          1 non-null      object 
 5   author_flair_text              0 non-null      float64
 6   author_flair_text_color        0 non-null      float64
 7   author_flair_type              1 non-null      str    
 8   brand_safe                     1 non-null      bool   
 9   can_gild                       1 non-null      bool   
 10  contest_mode                   1 non-null      bool   
 11  created_utc                    1 non-null      int64  
 12  distinguished                  0 non-null      float64
 13  domai

Zbiór zawierał 57 kolumn dla submissions będę potrzebować 12 kolumn:
- created_utc - daty
- subreddit - porównanie popheads i hiphopheads
- selftext (text dla submissions)
- title
- score - ilość upvotes
- author
- locked - czy została zablokowana możliwość komentowania
- gilded - ilość przyznanych płatnych wyróżnień
- id
- is_self - chcę zostawić tylko posty tekstowe
- permalink - link do posta
- num_comments - liczba komentarzy


Wyfiltrowane danych 1.02 - 31.03, wybrane 11 kolumn - komentarze:

In [ ]:
# start_period = int(datetime(2016, 2, 1).timestamp())
# end_period = int(datetime(2016, 3, 31, 23, 59, 59).timestamp())

# keep_columns = [
#     'link_id', 'controversiality', 'score', 'distinguished', 
#     'body', 'created_utc', 'subreddit', 'parent_id', 
#     'id', 'author', 'gilded'
# ]

# files_to_process = [
#     ("data/hiphopheads_comments", "data/hiphopheads_com_filtered_feb_march_2016.jsonl"),
#     ("data/popheads_comments", "data/popheads_com_filtered_feb_march_2016.jsonl")
# ]

# for input_file, output_file in files_to_process:  
#     with open(output_file, "w", encoding="utf-8") as out:
#         for chunk in pd.read_json(input_file, lines=True, chunksize=10000):
#             chunk["created_utc"] = pd.to_numeric(chunk["created_utc"], errors="coerce")
#             filtered = chunk[
#                 (chunk["created_utc"] >= start_period) & 
#                 (chunk["created_utc"] <= end_period)
#             ].copy()

#             existing_cols = [c for c in keep_columns if c in filtered.columns]
#             filtered = filtered[existing_cols]

#             for row in filtered.to_dict("records"):
#                 out.write(json.dumps(row) + "\n")

Submissions - daty 1.02-31.03, wyfiltrowane posty dla których is_self = True:

In [ ]:
# keep_columns_subs = [
#     'created_utc', 'subreddit', 'selftext', 'title', 
#     'score', 'author', 'locked', 'gilded', 
#     'id', 'is_self', 'permalink', 'num_comments'
# ]

# submissions_to_process = {
#     "data/popheads_submissions": "data/pop_subs_filtered_feb_march_2016.jsonl",
#     "data/hiphopheads_submissions": "data/hiphop_subs_filtered_feb_march_2016.jsonl"
# }

# for input_f, output_f in submissions_to_process.items():
#     with open(output_f, "w", encoding="utf-8") as out:
#         for chunk in pd.read_json(input_f, lines=True, chunksize=10000):
#             chunk["created_utc"] = pd.to_numeric(chunk["created_utc"], errors="coerce")
#             mask = (chunk["created_utc"] >= start_period) & (chunk["created_utc"] <= end_period)
#             filtered = chunk[mask].copy()

#             if 'is_self' in filtered.columns:
#                 filtered = filtered[filtered['is_self'] == True]

#             existing_cols = [c for c in keep_columns_subs if c in filtered.columns]
#             filtered = filtered[existing_cols]

#             for row in filtered.to_dict("records"):
#                 out.write(json.dumps(row) + "\n")

Przetwarzam submissions: data/popheads_submissions_2016.jsonl...
Przetwarzam submissions: data/hiphopheads_submissions_2016.jsonl...


Pre-processing przygotowanie funkcji:
- usunięcie linków
- usunięcie znaków interpunkcyjnych oprócz ! ? ' ' - które mogą nieść emocje oraz apostrofów np don't
- usunięcie liczb
- usunięcie podwójnych spacji

- podjęto decyzję o NIE usuwaniu liczb

In [ ]:
def clean_reddit_text(text):
    if not isinstance(text, str):
        return ""
        
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s!?\']', '', text)
    text = " ".join(text.split())
    
    return text

Wczytanie danych do zmiennych:

In [28]:
df_pop_subs = pd.read_json("data/pop_subs_filtered_feb_march_2016.jsonl", lines=True)
df_hip_subs = pd.read_json("data/hiphop_subs_filtered_feb_march_2016.jsonl", lines=True)
df_pop_com = pd.read_json("data/popheads_com_filtered_feb_march_2016.jsonl", lines=True)
df_hip_com = pd.read_json("data/hiphopheads_com_filtered_feb_march_2016.jsonl", lines=True)

Sprawdzenie czy dane nie zajmują zbyt dużo miejsca w pamięci:

In [29]:
for name, df in [("Pop Subs", df_pop_subs), ("Hip Subs", df_hip_subs), 
                 ("Pop Com", df_pop_com), ("Hip Com", df_hip_com)]:
    print(f"{name}: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB, Wierszy: {len(df)}")

Pop Subs: 0.27 MB, Wierszy: 300
Hip Subs: 3.81 MB, Wierszy: 6564
Pop Com: 7.70 MB, Wierszy: 14272
Hip Com: 246.27 MB, Wierszy: 500519


Wywołanie funkcji clean_reddit text na polach "title", "selftext" oraz "body"

In [ ]:
df_pop_subs['cleaned_title'] = df_pop_subs['title'].apply(clean_reddit_text)
df_pop_subs['cleaned_selftext'] = df_pop_subs['selftext'].apply(clean_reddit_text)

df_hip_subs['cleaned_title'] = df_hip_subs['title'].apply(clean_reddit_text)
df_hip_subs['cleaned_selftext'] = df_hip_subs['selftext'].apply(clean_reddit_text)

df_pop_com['cleaned_body'] = df_pop_com['body'].apply(clean_reddit_text)
df_hip_com['cleaned_body'] = df_hip_com['body'].apply(clean_reddit_text)


Czyszczenie postów (submissions)...
Czyszczenie komentarzy (comments)...
Gotowe! Dane zostały wyczyszczone.

Przykład po czyszczeniu (HipHopHeads Comments):
                                                body  \
0  He got shot and drove to the hospital in his L...   
1                                  Who tweets these?   
2     Same here dude. Even NWTS Drake was better imo   
3                              A fambruh's wet dream   
4    Lmao I didn't mean artists with only one ALBUM.   

                                        cleaned_body  
0  he got shot and drove to the hospital in his l...  
1                                  who tweets these?  
2      same here dude even nwts drake was better imo  
3                              a fambruh's wet dream  
4     lmao i didn't mean artists with only one album  


Usunięcie stopwords, ale pozostawienie słów przeczących. Do stopwords z biblioteki stopwords dla języka angielskiego dodanie stopwords charakterytycznych dla reddita

In [41]:
nltk.download("stopwords")

def remove_my_stopwords(text):
    
    stop_words = set(stopwords.words("english"))
    negations = {'not', 'no', 'never', 'but', 'however', 'isn', "isn't", 'wasn', "wasn't", 'don', "don't"}
    stop_words = stop_words - negations

    reddit_stop = ['deleted', 'removed', 'edit', 'edited', 'tldr', 'submission', 
                   'link', 'post', 'thread', 'comment', 'sub', 'subreddit', 'user', 
                   'username', 'gold', 'gilded', 'silver', 'platinum', 'thanks', 
                   'stranger', 'kind', 'lol', 'lmao', 'imo', 'imho', 'stfu', 
                   'afaik', 'fyi', 'btw', 'op']

    stop_words.update(reddit_stop)

    filtred_words = []
    for word in text.split():
        if word not in stop_words:
            filtred_words.append(word)
            
    return " ".join(filtred_words)

df_pop_com['final_body'] = df_pop_com['cleaned_body'].apply(remove_my_stopwords)
df_hip_com['final_body'] = df_hip_com['cleaned_body'].apply(remove_my_stopwords)

df_pop_subs['final_title'] = df_pop_subs['cleaned_title'].apply(remove_my_stopwords)
df_pop_subs['final_selftext'] = df_pop_subs['cleaned_selftext'].apply(remove_my_stopwords)

df_hip_subs['final_title'] = df_hip_subs['cleaned_title'].apply(remove_my_stopwords)
df_hip_subs['final_selftext'] = df_hip_subs['cleaned_selftext'].apply(remove_my_stopwords)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Konwersja czasu na format datetime - utworzenie dwóch nowych kolumn
Kolumna - timestamp - data wraz z czasem
Kolumna date - sama data bez czasu

In [ ]:
df_hip_com['timestamp'] = pd.to_datetime(df_hip_com['created_utc'], unit='s')
df_pop_com['timestamp'] = pd.to_datetime(df_pop_com['created_utc'], unit='s')
df_hip_subs['timestamp'] = pd.to_datetime(df_hip_subs['created_utc'], unit='s') 
df_pop_subs['timestamp'] = pd.to_datetime(df_pop_subs['created_utc'], unit='s') 

df_hip_com['date'] = df_hip_com['timestamp'].dt.date
df_pop_com['date'] = df_pop_com['timestamp'].dt.date 
df_hip_subs['date'] = df_hip_subs['timestamp'].dt.date 
df_pop_subs['date'] = df_pop_subs['timestamp'].dt.date

display(df_hip_com[['created_utc', 'timestamp', 'date']].head(1))

--- HipHopHeads Comments ---


,created_utc,timestamp,date
0,1454281212,2016-01-31 23:00:12,2016-01-31



--- Popheads Comments ---


,created_utc,timestamp,date
0,1454281642,2016-01-31 23:07:22,2016-01-31



--- HipHopHeads Submissions ---


,created_utc,timestamp,date
0,1454281899,2016-01-31 23:11:39,2016-01-31



--- Popheads Submissions ---


,created_utc,timestamp,date
0,1454281557,2016-01-31 23:05:57,2016-01-31


Lematyzacja: